**DS607 Group Proj**

Step 1 : Data collection and cleaning

Step 2 : Construct the sentiment/volatility variable

Step 3 : Descriptive exploration

Step 4 : Baseline regression

Step 5 : Hypothesis testing on β

Step 6 : Bootstrap CI (block bootstrap)

Step 7 : Distributional check / MLE

Step 8 : High/low-VIX regime comparison

---
*Refactored: reusable logic now lives in `src/`; this notebook calls those functions and shows results/narrative.*


**Setup**

Resolve project paths (relative to the repo root — works locally, in VS Code, or in Colab after cloning, no Drive dependency) and import project modules from `src/`.

In [ ]:
import os
import sys

# PROJECT_ROOT resolves relative to this notebook's location. Assumes the
# standard repo layout (project-repository/notebooks/this_notebook.ipynb) and
# that the notebook is run from within notebooks/, which is the default
# working directory when opened via Jupyter, VS Code, or Colab-after-clone.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)  # lets `import src` find src/ at the repo root

RAW_DIR = f"{PROJECT_ROOT}/data/raw"
INTERIM_DIR = f"{PROJECT_ROOT}/data/interim"
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"
FIG_DIR = f"{PROJECT_ROOT}/outputs/figures"
TABLE_DIR = f"{PROJECT_ROOT}/outputs/tables"

In [ ]:
from src import data_prep, descriptive, models, bootstrap, regime, nonparametric

print("Modules loaded:", data_prep.__name__, descriptive.__name__,
      models.__name__, bootstrap.__name__, regime.__name__, nonparametric.__name__)

**Step 1**

Data preparation
- Resample daily VIX to month-end value, aligned to CRSP's month-end convention
- Construct dividend yield control: `DIVₜ = MthTotRetₜ − MthPrcRetₜ`
- Handle the 4 non-month-end VIX gaps (confirm no impact; document the fix used, e.g. forward-fill or drop)
- Lag VIX and DIV by one month to align with `Rₜ₊₁` (next-month return)
- Merge into a single analysis-ready monthly dataset (`data/processed/merged_monthly.csv`)
- Construct High/Low-VIX regime indicator (median split or VIX = 20 threshold)
- Save intermediate/working files (e.g. resampled VIX before merge) to `data/interim/`


In [ ]:
df = data_prep.build_dataset(RAW_DIR, INTERIM_DIR, PROCESSED_DIR)
print("Final dataset shape:", df.shape)
df.head()

**Step 2**

Descriptive Analysis

- Time series plot: VIX and S&P 500 returns over 2001–2025
- Return histogram / Q-Q plot (check non-normality)
- VIX-vs-next-month-return scatter plot
- Summary statistics table (mean, SD, min/max) for returns, VIX, dividend yield
- Correlation / VIF between VIXₜ and DIVₜ (multicollinearity check)


In [ ]:
descriptive.plot_timeseries(df, FIG_DIR)
sw_stat, sw_p = descriptive.plot_histogram_qq(df, FIG_DIR)
print(f"Shapiro-Wilk: W={sw_stat:.4f}, p={sw_p:.4g}")

descriptive.plot_scatter(df, FIG_DIR)

In [ ]:
summary_table = descriptive.summary_statistics_table(df, TABLE_DIR)
display(summary_table.round(5))

corr, vif_data = descriptive.vif_check(df, TABLE_DIR)
print(f"Correlation (VIX, DIV): {corr:.4f}")
display(vif_data)

**Step 3**

Baseline regression (VIX only)

- Estimate `Rₜ₊₁ = α + β·VIXₜ + εₜ` via `statsmodels.OLS`
- Report β, SE, t-stat, p-value, R²
- Test H₀: β = 0 vs H₁: β ≠ 0 (df = n − 2)


In [ ]:
baseline_results = models.fit_baseline(df)
print(baseline_results.summary())

baseline_test = models.baseline_hypothesis_test(baseline_results)
print(f"H0: beta = 0 -> t={baseline_test['t_stat']:.4f}, p={baseline_test['p_value']:.4g}, "
      f"df={baseline_test['df']} -> {'Reject' if baseline_test['reject_h0'] else 'Fail to reject'} H0")

baseline_table = models.save_baseline_table(baseline_results, TABLE_DIR)
display(baseline_table)

**Step 4**

Extended regression (VIX + dividend yield)
- Estimate `Rₜ₊₁ = α + β₁·VIXₜ + β₂·DIVₜ + εₜ`
- Report β₁, β₂, SE, t-stats, p-values, adjusted R² (not R²)
- Test H₀: β₁ = 0 and H₀: β₂ = 0 separately (df = n − 3)
- Run F-test for joint significance (H₀: β₁ = β₂ = 0)
- Report ordinary OLS SEs **and** HAC/robust SEs (e.g. Newey-West) side by side
- Present baseline vs. extended model results in one comparison table


In [ ]:
ext_ols, ext_hac = models.fit_extended(df)
print(ext_ols.summary())
print(ext_hac.summary())

In [ ]:
ext_tests = models.extended_hypothesis_tests(ext_ols, ext_hac)
print(f"df = {ext_tests['df']}, Adjusted R2 = {ext_tests['adj_r2']:.4f}")
print(f"F-test (joint): F={ext_tests['f_stat']:.4f}, p={ext_tests['f_pvalue']:.4g} -> "
      f"{'Reject' if ext_tests['reject_joint'] else 'Fail to reject'} H0")

comparison_table = models.save_extended_comparison_table(baseline_table, ext_ols, ext_hac, TABLE_DIR)
display(comparison_table.round(6))

**Step 5**

Block-bootstrap inference
- Implement moving block-bootstrap (following Schmeling, 2009)
- For the extended model, resample full rows (VIXₜ, DIVₜ, Rₜ₊₁) jointly to preserve time-series dependence
- Compute bootstrap confidence intervals for β₁ and β₂
- Compare bootstrap CIs against asymptotic (OLS/HAC) CIs; discuss any divergence
- Save bootstrap distribution plot to `outputs/figures/`


In [ ]:
boot_beta1, boot_beta2, block_len = bootstrap.run_block_bootstrap(df, n_boot=5000, seed=42)
print(f"Block length: {block_len}")

ci_beta1 = bootstrap.bootstrap_ci(boot_beta1)
ci_beta2 = bootstrap.bootstrap_ci(boot_beta2)
print(f"Bootstrap 95% CI beta1 (VIX): [{ci_beta1[0]:.6f}, {ci_beta1[1]:.6f}]")
print(f"Bootstrap 95% CI beta2 (DIV): [{ci_beta2[0]:.6f}, {ci_beta2[1]:.6f}]")

In [ ]:
ci_table = bootstrap.save_ci_comparison_table(ext_ols, ext_hac, ci_beta1, ci_beta2, TABLE_DIR)
display(ci_table.round(6))

bootstrap.plot_bootstrap_distributions(
    boot_beta1, boot_beta2, ci_beta1, ci_beta2,
    ext_ols.params["vix_level"], ext_ols.params["div_yield"], FIG_DIR
)

**Step 6**

High- vs. low-VIX regime comparison

- Split sample at median VIX (or VIX = 20) into high-fear / low-fear months
- Two-sample t-test comparing mean next-month returns across regimes
- Check subgroup sizes are large enough given the extended 2001–2025 sample


In [ ]:
result_median = regime.regime_ttest(df, "regime_median", "Median VIX split")
result_threshold = regime.regime_ttest(df, "regime_threshold20", "VIX = 20 threshold split")

regime_table = regime.save_regime_table([result_median, result_threshold], TABLE_DIR)
display(regime_table.round(5))

regime.plot_regime_boxplots(df, FIG_DIR)

**Step 7**

Nonparametric test: permutation test on β

- H0: no association between predictors and next-month returns (permute R_{t+1} relative to VIX/DIV to build an empirical null distribution)
- Report permutation p-values for baseline β and extended β1, β2
- Compare against OLS p-values from Steps 3–4


In [ ]:
perm_result = nonparametric.run_permutation_test(df, n_perm=5000, seed=42)

print("Observed estimates:", perm_result["observed"])
print("Permutation p-values:", perm_result["p_values"])
print("OLS p-values (for comparison):", perm_result["ols_p_values"])

In [ ]:
perm_table = nonparametric.save_permutation_table(perm_result, TABLE_DIR)
display(perm_table.round(5))

nonparametric.plot_permutation_distributions(perm_result, FIG_DIR)

**Confirm outputs saved**

In [ ]:
import os
print("Figures:", os.listdir(FIG_DIR))
print("Tables:", os.listdir(TABLE_DIR))